# Lab 12 — ANSWER KEY — NumPy, Pandas, and the Great American Format War

**CSC 2053 — Fall 2026**

In this lab you will:
- Learn the basics of **NumPy** — the foundation of numerical computing in Python
- Learn the basics of **Pandas** — the go-to library for tabular data
- Apply both to real PPM ratings data and propose format changes for one U.S. radio market

By the end you will play the role of a programming consultant pitching three format flips to the FCC Chair (me). The student whose pitch adds the most projected audience share wins.

> **Save often** — use **Ctrl+S** (Windows) or **Cmd+S** (Mac) as you go.
> When finished, download via **File → Download → Download .ipynb** and submit to your course GitHub repository.

---

In [ ]:
# Run this cell first — imports everything you need
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

---
## Part 1: NumPy Basics

**NumPy** (Numerical Python) gives you fast, memory-efficient arrays and a huge library of math operations. Instead of looping over a list, you operate on the *whole array at once*.

### 1.1 Creating Arrays

In [ ]:
# NumPy arrays are created with np.array()
# Here are some sample FM frequencies (MHz)
frequencies = np.array([88.1, 90.9, 94.5, 99.5, 101.1, 104.3, 107.9])

print("Frequencies:", frequencies)
print("Type:", type(frequencies))
print("Shape:", frequencies.shape)   # (7,) means 7 elements, 1 dimension
print("Data type:", frequencies.dtype)

In [ ]:
# YOUR TURN: Create a NumPy array called `powers` with these ERP values (in kW):
# 0.25, 3.0, 50.0, 100.0, 6.0, 25.0, 0.1

powers = np.array([0.25, 3.0, 50.0, 100.0, 6.0, 25.0, 0.1])

print("Powers (kW):", powers)

### 1.2 Array Math

One of NumPy's biggest advantages: **math applies to every element at once**, no loop needed.

In [ ]:
# Convert kW to Watts by multiplying the entire array by 1000
powers_watts = powers * 1000
print("Powers (W):", powers_watts)

In [ ]:
# YOUR TURN:
# FM stations in the U.S. broadcast between 87.9 and 107.9 MHz.
# Convert `frequencies` from MHz to kHz (multiply by 1000).
# Store the result in `frequencies_khz` and print it.

frequencies_khz = frequencies * 1000
print("Frequencies (kHz):", frequencies_khz)

### 1.3 Boolean Indexing

You can filter an array using a condition — this returns a *boolean mask* (True/False for each element), which you then use to select values.

In [ ]:
# Find all frequencies above 100 MHz
mask = frequencies > 100
print("Mask:", mask)
print("Above 100 MHz:", frequencies[mask])

In [ ]:
# YOUR TURN: Using the `powers` array, find all values with ERP greater than 10 kW.
# Print both the boolean mask and the filtered values.

high_power_mask = powers > 10
high_power_values = powers[high_power_mask]

print("Mask:", high_power_mask)
print("Values above 10 kW:", high_power_values)

### 1.4 Aggregation Functions

NumPy has built-in functions to summarize an array quickly.

In [ ]:
# Summary statistics on the powers array
print("Count  :", powers.size)
print("Sum    :", np.sum(powers), "kW")
print("Mean   :", np.mean(powers), "kW")
print("Median :", np.median(powers), "kW")
print("Min    :", np.min(powers), "kW")
print("Max    :", np.max(powers), "kW")
print("Std Dev:", np.std(powers).round(2), "kW")

In [ ]:
# YOUR TURN:
# 1. What is the average frequency in `frequencies`?
# 2. What is the highest frequency?
# 3. How many frequencies are there total?

avg_freq   = np.mean(frequencies)
max_freq   = np.max(frequencies)
count_freq = frequencies.size

print(f"Average frequency : {avg_freq} MHz")
print(f"Highest frequency : {max_freq} MHz")
print(f"Number of stations: {count_freq}")

### Part 1 Challenge

Using `np.arange()`, create an array of every valid FM frequency in the U.S. (88.1 to 107.9 MHz in steps of 0.2 MHz). Then answer: **how many possible FM channels are there?**

*Hint: `np.arange(start, stop, step)` works like Python's `range()` but for floats. The `stop` is exclusive, so to include 107.9 you need to go slightly beyond it.*

In [ ]:
fm_channels = np.arange(88.1, 108.0, 0.2).round(1)
print("FM channels:", fm_channels)
print("Total possible FM channels:", fm_channels.size)

---
## Part 2: Pandas Basics — Warming Up in Philadelphia

**Pandas** builds on NumPy to give you a **DataFrame** — think of it as a spreadsheet in Python. Each column is a **Series** (like a labeled NumPy array), and you can filter, group, sort, and summarize data with just a few lines.

For Part 2, we'll all warm up with **Philadelphia** radio stations. This is our full station roster for the Philly metro, with PPM (Portable People Meter) ratings merged in from RadioInsight.

### 2.1 Loading and Exploring Data

In [ ]:
# Load the full five-market station+ratings CSV
URL = "https://raw.githubusercontent.com/W2NJL/csc2053-fall26-labs/main/lab12/stations_rated.csv"
df = pd.read_csv(URL)

print("Shape:", df.shape)
df.head()

In [ ]:
# .info() shows column names, data types, and non-null counts
df.info()

In [ ]:
# .describe() gives summary statistics for every numeric column
df.describe()

### 2.2 Filtering

Filtering a DataFrame uses a boolean condition inside `[]`, just like NumPy boolean indexing.

In [ ]:
# Filter to Philadelphia stations only
philly = df[df['market'] == 'Philadelphia']
print("Total Philadelphia stations:", len(philly))
philly[['callsign', 'band', 'format', 'parent_group']].head()

In [ ]:
# YOUR TURN: Filter `philly` to only FM stations and store in `philly_fm`.
# Then print how many FM stations are in Philadelphia.

philly_fm = philly[philly['band'] == 'FM']
print("FM stations in Philadelphia:", len(philly_fm)
      )

### 2.3 Combining Conditions with `&` and `|`

When you filter a DataFrame, each condition produces an entire column of True/False values — one per row. Plain Python's `and`/`or` can only compare two single booleans, so for row-by-row logic across a Series we use **`&` (and)** and **`|` (or)**.

Wrap each condition in parentheses — it's required.

In [ ]:
# Example: Philadelphia FM stations that have a measured PPM rating
rated_fm = philly[(philly['band'] == 'FM') & (philly['avg_share_6mo'].notna())]
print("Philly FM stations with a rating:", len(rated_fm))
rated_fm[['callsign', 'format', 'parent_group', 'avg_share_6mo']].head()

**Note on messy data:** Not every station in the file has a rating. RadioInsight publishes the top ~25-30 stations per market; smaller and non-commercial stations fall below the cutoff and show up with `NaN` for their share. Our Pandas filter with `.notna()` lets us work with only the rated stations.

This is what real analyst work looks like — data has gaps, and you decide how to handle them.

In [ ]:
# YOUR TURN: Find Philadelphia stations with a rating above 3.0 share.
# Hint: combine two conditions with & — one for market, one for share.
# Display callsign, band, format, parent_group, avg_share_6mo.

top_philly = philly[(philly['market'] == 'Philadelphia') & (philly['avg_share_6mo'] > 3.0)]
print("Philly stations above 3.0 share:", len(top_philly))
top_philly[['callsign', 'band', 'format', 'parent_group', 'avg_share_6mo']]

### 2.4 Counting with `.value_counts()`

`.value_counts()` tells you how many times each unique value appears in a column.

In [ ]:
# Count of each format in Philadelphia
philly['format'].value_counts().head(10)

In [ ]:
# YOUR TURN: Use .value_counts() on `parent_group` to find the top 5
# station-owning companies in Philadelphia.

philly['parent_group'].value_counts().head(5)

### 2.5 Grouping with `.groupby()`

`.groupby()` splits a DataFrame into groups and lets you compute a statistic per group.

In [ ]:
# Sum the measured share of all Philadelphia stations grouped by format.
# This tells us how much of Philly's listening each format captures in total.
philly_format_shares = philly.groupby('format')['avg_share_6mo'].sum().sort_values(ascending=False)
philly_format_shares.head(10)

In [ ]:
# YOUR TURN: Use .groupby() on `parent_group` to compute the TOTAL rated
# share each ownership company has in Philadelphia.
# Sort descending.

philly[philly['avg_share_6mo'].notna()].groupby('parent_group')['avg_share_6mo'].sum().sort_values(ascending=False)

### 2.6 Sorting

In [ ]:
# Sort Philadelphia rated stations by share to find the top performers
philly[philly['avg_share_6mo'].notna()].sort_values('avg_share_6mo', ascending=False)[
    ['callsign', 'format', 'parent_group', 'avg_share_6mo']
].head(10)

### Part 2 Challenge

In **one chain of Pandas calls** (no intermediate variables), find the **top 3 ownership companies** by total rated share in Philadelphia.

*Hint: filter to rated stations, groupby parent_group, sum the share, sort, head.*

In [ ]:
philly[philly['avg_share_6mo'].notna()].groupby('parent_group')['avg_share_6mo'].sum().sort_values(ascending=False).head(3)

---
## Part 3: The Brendan Carr Hearing — Format War

**Scenario:** I'll play FCC Chair Brendan Carr. You are a programming consultant. You have been assigned one of our five metered markets and must propose **three format flips** — taking three existing stations, switching their format, and projecting the audience gain.

The student whose three flips produce the highest **total projected score** wins the hearing.

### Industry Context: Why Cumulus Isn't Here

You may notice that **Cumulus Media** stations (e.g. WLS-FM Chicago, WMAL-FM DC) don't show up in the ratings data. That's because Cumulus pulled out of Nielsen measurement during their ongoing legal dispute. Any station not subscribed to Nielsen is invisible to the PPM data, even if it's a real heritage station. This is a real industry situation — measurement matters.

### The Rules

1. **AM stations cannot flip to a music format.** In 2026, AM music stations don't compete — the audience has long since moved on. AM stations can only flip to **News/Talk** or **Sports**. Any AM proposal targeting a music format scores **zero**.
2. **Noncommercial stations cannot be flip targets.** Public, university, and religious noncommercial stations (reserved-band FMs 88.1–91.9 MHz, plus commercial-band noncoms like WNYC-FM and WQXR-FM) are mission-driven, not revenue-driven — they don't flip for ratings. They stay in the data (their share counts toward the market total), but you cannot propose one as a flip.
3. All three flips are mandatory.
4. **Opportunity cost matters.** Flipping a station means abandoning its current audience. If the projected post-flip share is *smaller* than what the station already has, it's a bad flip and scores zero.
5. **Signal coverage matters.** A station that doesn't put a strong signal into the metro center can't realistically capture a full-market format. Each station's predicted field strength at the metro center (in dBuV/m, computed by the Judy FM propagation model / FCC AM groundwave model) is in the `dbuv_at_center` column. 90+ = full coverage; 80–90 = partial; below 80 = fringe.
6. The pitch paragraph in Step 8 must justify your flips with reference to **Format Index, Demographic Composition Index, Signal Coverage, and (if applicable) the AM constraint.**

### The Scoring Formula

For each flip we compute a **projected post-flip share** using four multiplicative factors, then subtract the station's current audience as opportunity cost:

$$\text{projected\_share} = \text{national\_share} \times \max\!\left(0,\ 1 - \tfrac{\text{Format Index}}{100}\right) \times \text{Demographic Lift} \times \text{Signal Weight}$$

$$\text{flip\_score} = \max(0,\ \text{projected\_share} - \text{current\_share})$$

Where:
- **`national_share`** = Nielsen P25-54 AQH share for the target format in PPM markets (from Inside Radio's August 2026 analysis). P25-54 is the "money demo" — the age range advertisers pay the most for.
- **`Format Index`** = `(market_share ÷ national_share) × 100`. Below 100 → room to grow.
- **`Demographic Lift`** = `local_demo_% ÷ national_demo_%` for the demographic the format targets:
  - **Urban AC** and **Urban Contemporary** → Black composition (national = 13.6%)
  - **Spanish Contemporary** → Hispanic composition (national = 19.5%)
  - All other formats default to 1.0.
- **`Signal Weight`** = full credit (1.0) at ≥ 90 dBuV/m into the metro center; linear falloff to 0.5 at 80 dBuV/m; steep drop below.
- **`current_share`** = what the station already captures.

Your **total pitch score** is the sum of your three flip scores.

**Why Mexican Regional isn't on the board:** Nielsen reports Spanish Regional Mexican at 3.6 share (P25-54) nationally, but it's a Southwest/Border format — none of our five metros (NY, Philly, Boston, Miami, SF) have the Mexican-American demographic core to realistically support it. We've excluded it from the target list.

**Citations:** Inside Radio analysis of Nielsen Audio PPM AQH shares, August 2026 survey (P25-54); TVB *Research 101* (indexing methodology); FCC M3 Map / Longley-Rice foundations for FM propagation; Sweeting, A. (2013), *Econometrica* 81(5):1763–1803 (discrete-choice format decisions).

---

### Step 1 — Set Your Market

I'll assign you one of: `Philadelphia`, `Boston`, `New York`, `Miami`, or `San Francisco`.

In [ ]:
MARKET = "New York"   # <-- Replace with your assigned market name (exact spelling)

# Filter to your market's RATED stations only
market_data = df[(df['market'] == MARKET) & (df['avg_share_6mo'].notna())].copy()
print(f"Rated stations in {MARKET}: {len(market_data)}")

# dbuv_at_center is the predicted field strength (in dBuV/m) at the metro's
# geographic center, from the Judy FM propagation model (and the FCC AM
# groundwave model for AMs). 90+ = full metro coverage. 80–90 = partial.
# Below 80 = fringe signal; such stations only made the dataset if they
# already rate in the market.
market_data[['callsign', 'band', 'format', 'format_nielsen',
             'parent_group', 'avg_share_6mo', 'dbuv_at_center']] \
    .sort_values('avg_share_6mo', ascending=False).head(15)

### Step 1a — Know Your Market's Audience

Who actually lives in your market? Run the cell below to see the Nielsen TSA population and demographic breakdown.

**Read the percentages, not the raw counts.** A city can have a *big absolute* minority population and still *under-index* against the national average — e.g. Boston's ~394,000 Black residents is a sizable community, but at ~8.7% of a 4.5M metro it's actually *below* the U.S. average of 13.6%. By contrast, Philadelphia's Black share is 20.5% — significantly above average — so an Urban AC format has a much stronger natural audience there than in Boston.

The code below prints your market's demographics *alongside the U.S. national benchmark* so you can tell at a glance whether each group over- or under-indexes. You'll reference this in your pitch (Step 8).

In [ ]:
MARKETS_URL = "https://raw.githubusercontent.com/W2NJL/csc2053-fall26-labs/main/lab12/markets.csv"
markets = pd.read_csv(MARKETS_URL)

# U.S. 2024 Census benchmarks — use these to decide whether your market
# OVER-INDEXES or UNDER-INDEXES on each demographic
US_HISPANIC_PCT = 19.5   # national Hispanic share of population
US_BLACK_PCT    = 13.6   # national Black share of population

print("All 5 markets in the hearing:")
print(markets.to_string(index=False))
print()
print(f"(National benchmarks: Hispanic {US_HISPANIC_PCT}%, Black {US_BLACK_PCT}%)")
print()

my_demo = markets[markets['market'] == MARKET].iloc[0]
hisp_delta = my_demo['hispanic_pct'] - US_HISPANIC_PCT
black_delta = my_demo['black_pct']    - US_BLACK_PCT

def _label(delta):
    if delta >=  3.0: return "ABOVE national avg"
    if delta <= -3.0: return "BELOW national avg"
    return "near national avg"

print(f"--- Your market: {MARKET} ---")
print(f"  Nielsen rank       : #{my_demo['rank']}")
print(f"  Population (12+)   : {my_demo['population_12plus']:,}")
print(f"  Hispanic share     : {my_demo['hispanic_pct']:>5}%   ({hisp_delta:+.1f} pts vs US — {_label(hisp_delta)})")
print(f"  Black share        : {my_demo['black_pct']:>5}%   ({black_delta:+.1f} pts vs US — {_label(black_delta)})")
print()
print("Reading the context:")
print("  - Raw population counts can mislead (Boston's ~394K Black residents is a big number,")
print("    but at 8.7% of a 4.5M market it's BELOW the U.S. average).")
print("  - A format targeted to a demographic over-performs its national share when the")
print("    market's share of that group over-indexes, and vice versa.")

### Step 2 — Compute Your Market's Current Format Landscape

Our dataset actually has **three** format columns:
- `format` — what the FCC and RadioLand list the station as (very granular; e.g. `Rhythmic CHR`, `Adult R&B`, `News/Talk/Jazz`)
- `format_ri` — what RadioInsight categorizes the station as (closer to Nielsen but not identical)
- `format_nielsen` — the **canonical Nielsen top-15 category** (e.g. `Urban AC`, `Contemporary Hit Radio`). This is the one that lines up with Nielsen's share-of-ear report.

We'll use **`format_nielsen`** for the hearing. Group your market's rated stations by `format_nielsen` and sum `avg_share_6mo` — that tells you how much of the market's measured listening each Nielsen format category captures.

In [ ]:
# YOUR TURN: build market_format_shares as a Series of format_nielsen -> total share.
# Sort descending so the biggest formats appear first.

market_format_shares = market_data.groupby('format_nielsen')['avg_share_6mo'].sum().sort_values(ascending=False)
market_format_shares

### Step 3 — Load Nielsen's National Format Shares

This is Inside Radio's analysis of Nielsen's **August 2026 PPM survey** (July 23–Aug 19) — total-week AQH shares in PPM markets. Two columns:

- **`p2554_share`** — persons 25-54, the "money demo" advertisers pay most for.
- **`p1834_share`** — persons 18-34.

Adult Contemporary leads both demos (7.2 P25-54, 7.9 P18-34). We'll use P25-54 for the gap analysis since that's the demo station owners optimize format decisions around.

In [ ]:
NIELSEN_URL = "https://raw.githubusercontent.com/W2NJL/csc2053-fall26-labs/main/lab12/nielsen_shares.csv"
nielsen = pd.read_csv(NIELSEN_URL)
nielsen

### Step 4 — Compute Format Gaps

For each Nielsen format, compare the national share-of-ear to what's actually captured in your market. Build a table `gaps` with columns `format`, `nielsen`, `market_actual`, `gap`.

`gap = nielsen_share - market_actual_share`. Positive gap = under-served in your market = a good flip target.

In [ ]:
# Build the gaps table. Start with the Nielsen P25-54 shares and look up each
# format's total share in your market (0 if the format isn't present at all).

gaps = nielsen[['format', 'p2554_share']].rename(columns={'p2554_share': 'nielsen'}).copy()

# For each format, pull its summed market share (fill 0 if absent)
gaps['market_actual'] = gaps['format'].map(market_format_shares).fillna(0).round(2)

# YOUR TURN: add the 'gap' column
gaps['gap'] = gaps['nielsen'] - gaps['market_actual']

gaps.sort_values('gap', ascending=False)

### Step 5 — Identify Your Top 3 Underserved Formats

Sort `gaps` by `gap` descending and take the top 3. These are the format targets most worth flipping into.

In [ ]:
# YOUR TURN: build `top_targets` — the 3 formats with the biggest positive gap

top_targets = gaps.sort_values('gap', ascending=False).head(3)
top_targets

### Step 6 — Pick Your Flip Candidates

For each of your three target formats, choose one existing rated station in your market to flip.

**Strategy tips — read these before you pick:**
- **Don't flip stalwarts.** Because opportunity cost is subtracted from the score, flipping a 7-share leader (think WLTW or WINS in NY) is almost always a losing move — their current audience is worth more than any realistic projected flip share. In real life, PDs flip *mid-performing* stations in *oversaturated* formats. The model rewards you for doing the same.
- **AM stations are restricted.** They can only flip to News/Talk, Sports, or News. Any AM flip to a music format scores zero.
- **Demographic composition matters for Urban AC, Urban Contemporary, and Mexican Regional.** In a market that over-indexes on the relevant group, those formats get a multiplicative lift. In a market that under-indexes, they get penalized. Check your Step 1a output.
- The `format_nielsen` spelling must match one in the `gaps` table *exactly*.

Look back at your `market_data` listing above. Decide three stations, then fill in the proposals below.

In [ ]:
# YOUR PROPOSALS: fill in (callsign, new_format) for each of your three flips.
# The new_format must appear in the `gaps` table — use the exact spelling.

proposals = [
    ("WEPN-FM", "Country"),        # Hot AC flagship, NY has no Country station
    ("WXNY-FM", "Alternative"),    # weak Spanish Contemp, Alt is under-indexed
    ("WXBK",    "Adult Hits"),     # mid-share Urban Contemp, Adult Hits untapped
]
proposals

### Step 7 — Score Your Flips

Each proposal is scored by the formula from the Part 3 intro — projected post-flip share minus opportunity cost, floored at zero.

You'll write three things:

1. **`demographic_lift(target_format)`** — a short function that returns the Composition Index for the demographic a format targets (or `1.0` if it targets no specific group). Three `if` branches.
2. **Station lookup** inside `score_flip` — one line filtering `market_data` to the row for the given callsign.
3. **Target format lookup** inside `score_flip` — one line filtering the `gaps` table to the row for the target format.

Everything else (the AM ban, the signal weight, the arithmetic) is given.

Run the cell to see each flip's current share, projected share, signal strength, and net score.

In [ ]:
# National demographic benchmarks (same values you saw in Step 1a)
US_HISPANIC_PCT = 19.5
US_BLACK_PCT    = 13.6

# Which Nielsen formats are targeted to each demographic group
BLACK_FORMATS    = {"Urban AC", "Urban Contemporary"}
HISPANIC_FORMATS = {"Spanish Contemporary"}

# AM stations may only flip to spoken-word formats (News/Talk now bundles News per Inside Radio)
AM_ALLOWED_FORMATS = {"News/Talk", "Sports"}


def demographic_lift(target_format):
    """
    Composition Index for the demographic a format targets.
    Formats without a demographic target get 1.0.
    """
    # YOUR TURN — three cases:
    #   target_format in BLACK_FORMATS     → return my_demo['black_pct']    / US_BLACK_PCT
    #   target_format in HISPANIC_FORMATS  → return my_demo['hispanic_pct'] / US_HISPANIC_PCT
    #   otherwise                          → return 1.0

    if target_format in BLACK_FORMATS:
        return my_demo['black_pct'] / US_BLACK_PCT
    if target_format in HISPANIC_FORMATS:
        return my_demo['hispanic_pct'] / US_HISPANIC_PCT
    return 1.0


def signal_weight(dbuv):
    """
    Propagation penalty for stations that don't cover the metro center well.
    >= 90 dBuV/m → full strength (1.0). Linear falloff to 0.5 at 80 dBuV/m.
    Below 80 → steep drop.  NaN → no penalty (unknown signal, assume full).
    """
    if pd.isna(dbuv) or dbuv >= 90:
        return 1.0
    if dbuv >= 80:
        return 0.5 + (dbuv - 80) / 20
    return max(0.1, 0.5 * (dbuv / 80))


def score_flip(callsign, target_format):
    # YOUR TURN: pull the station's row from market_data (a single-row DataFrame)
    station = market_data[market_data['callsign'] == callsign]

    # YOUR TURN: pull the target format's row from the gaps table (a single-row DataFrame)
    target  = gaps[gaps['format'] == target_format]

    band           = station.iloc[0]['band']
    current_share  = station.iloc[0]['avg_share_6mo']
    station_dbuv   = station.iloc[0]['dbuv_at_center']
    is_noncom      = station.iloc[0]['is_noncommercial']
    market_share   = target.iloc[0]['market_actual']
    national_share = target.iloc[0]['nielsen']

    # Noncom guardrail: public, university, religious stations aren't candidates
    if is_noncom:
        print(f"  {callsign:<9} {band}  →  {target_format:<22}  BLOCKED (noncommercial — not a flip candidate)")
        return 0.0

    # AM-to-music guardrail
    if band == "AM" and target_format not in AM_ALLOWED_FORMATS:
        print(f"  {callsign:<9} {band}  →  {target_format:<22}  BLOCKED (AM→music)")
        return 0.0

    # Format Index: 100 = at par with national.  Below 100 = headroom.
    format_index = (market_share / national_share) * 100
    headroom     = max(0.0, 1 - format_index / 100)

    # Projected post-flip share: national ceiling × headroom × demographic lift × signal coverage.
    sig_w = signal_weight(station_dbuv)
    projected_share = national_share * headroom * demographic_lift(target_format) * sig_w

    # Subtract opportunity cost (the audience the station leaves behind).
    net_delta = max(0.0, projected_share - current_share)

    print(f"  {callsign:<9} {band}  →  {target_format:<22}  "
          f"current={current_share:5.2f}  projected={projected_share:5.2f}  "
          f"sig={station_dbuv:5.1f}({sig_w:.2f})  net={net_delta:5.2f}")
    return net_delta


print("Flip-by-flip breakdown:")
total = sum(score_flip(cs, fmt) for cs, fmt in proposals)
print(f"\nTOTAL PITCH SCORE: {total:.2f}")

### Step 8 — Make Your Case

In the markdown cell below, write **3-4 sentences** explaining your pitch to the FCC. Your response must reference:
1. **Format Index logic** — which under-indexed format(s) you targeted and why
2. **Demographic Composition Index** — tie at least one flip to the Hispanic or Black composition of your market (from Step 1a), noting whether your market over- or under-indexes nationally
3. **Band constraint** — if any of your flips are AMs, say explicitly why that spoken-word target fits

> *Double-click the next cell to edit it.*

**My Pitch:**

*(Your 3-4 sentence case here. Reference share gap, demographics, and — if applicable — your AM-flip rationale.)*

---
## Submission

- Save your work — **Ctrl+S** / **Cmd+S**
- Before submitting, run all cells from the top: **Kernel → Restart & Run All**
- Confirm your `MARKET` is set correctly in Step 1
- Confirm your three proposals are filled in in Step 6
- Confirm your pitch paragraph is written in Step 8 and references **demographics** + **share gap**
- Download via **File → Download → Download .ipynb** and submit to your course GitHub repository

In [ ]:
your_name = ""   # First Last
print(f"Lab 12 submission — {your_name} — Market: {MARKET} — Total Score: {total:.2f}")